This cell imports all necessary packages

In [ ]:
# General imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import copy
import seaborn as sns

# Gensim imports
from gensim.corpora import Dictionary
from gensim.models import LdaModel
from gensim.models import HdpModel

# Sklearn imports
from sklearn.decomposition import LatentDirichletAllocation, TruncatedSVD, PCA
from sklearn.manifold import TSNE, MDS
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler

This code is intended for topic analysis and clustering on metagenomic datasets driven by Latent Dirichlet Allocation (LDA).

In particular, it takes collections of samples with corresponding OTU counts (for example - from 16S rRNA sequencing) and metadata.

It takes as input the following information organized as specified.
- OTU Count Table - Sample IDs in column 0, OTU frequency counts in subsequent columns.

    _Note: We do not recommend any sort of prescaling or prevalence/abundance filtering on the data. LDA is built to take "bag-of-words" style input._
- Sample Metadata - Sample IDs in column 0, relevant data in subsequent columns. 

Optionally, if you want to run similar processing at various taxonomic levels:
- Taxonomy Table - OTUs in column 0, taxonomic classifications (Order, Family, Genus, etc.) in subsequent columns.

Alternatively, you can just use a table that has counts of, i.e., Family frequency in place of the OTU count table.

Below is the code to generate the LDA model. LDA requires a pre-picked number of topics, so we first use HDP to generate an appropriate number of topics for the dataset.

There are a couple useful parameters to keep in mind:
> threshold:
>>HDP generates a huge number of topics. We prune all topics below a certain relevance threshold to get a topic number for LDA. Toggling this will give more generous or conservative topic counts. It is set to 0.01 by default.

>LDA_passes:
>>how many passes over the data LDA makes. A higher number means sounder analysis but longer runtime. It is set to 10 by default.

LDA priors:

LDA is a Bayesian inference technique. It makes assumptions about how the data is "generated" based on choice of $\alpha$ and $\eta$.

>$\alpha$ controls topics per document. A small $\alpha$ means spikier distributions, documents concentrate probability on fewer topics. A large $\alpha$ corresponds to a smoother distribution.

>$\eta$ controls words per topic. Similarly, a small $\eta$ means spikier topics focused on fewer words.

In general, small $\alpha$ and $\eta$ are preferable, leading to sparser, more interpretable topics. However, making these parameters too small increases risk of overfitting. The code lets Gensim automatically generate priors appropriate to the dataset, but this can be replaced by specificying a symmetric distribution or even manually choosing a distribution vector.

In [ ]:
def generateModels(file_name, threshold = 0.01, LDA_passes = 10):
    # load dataset
    df = pd.read_csv(file_name)
    df = df.drop(df.columns[0], axis=1)  # drop index column if present

    # convert frequency to bag-of-words
    texts = []
    for row in df.itertuples(index=False):
        doc = []
        for word, count in zip(df.columns, row):
            doc.extend([word] * int(count))
        texts.append(doc)

    # construct dictionary and corpus
    dictionary = Dictionary(texts)
    corpus = [dictionary.doc2bow(text) for text in texts]

    # get # topics from HDP
    hdp = HdpModel(corpus=corpus, id2word=dictionary)

    topic_usage = np.zeros(hdp.get_topics().shape[0])
    for doc in corpus:
        doc_topics = hdp[doc]  # list of (topic_id, prob)
        for topic_id, prob in doc_topics:
            topic_usage[topic_id] += prob
    topic_usage /= len(corpus) # gives the average "probability" of a topic across all documents

    topic_count = np.sum(topic_usage > threshold) # counts number of useful topics
    print(f"HDP inferred ~{topic_count} topics")

    lda = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=topic_count,
        random_state=0,
        passes=LDA_passes,
        alpha="auto", eta="auto" # toggle as desired
    )

    return lda, df, dictionary, corpus

The following function gives a breakdown of topics by their most important terms at the desired taxonomic level. For example, you might run LDA at the OTU level, but want output at the family level.

In [ ]:
def topicBreakdown(model, dictionary, taxonomy_file_name, input_taxonomy, output_taxonomy, output_path, term_num = 10):
    taxonomy = pd.read_csv(taxonomy_file_name)
    result = ''

    # create a map between your input and output levels for faster lookup
    taxonomy_map = dict(zip(taxonomy[input_taxonomy], taxonomy[output_taxonomy]))
    
    # get heaviest terms in each topic
    topic_ids = range(model.num_topics)
    get_terms = lambda i: model.get_topic_terms(i, topn=term_num)

    # iterate over topics
    for i in topic_ids:
        result += f"Topic {i}: "
        for term_id, weight in get_terms(i):
            word = dictionary[term_id]

            # match taxonomy
            result += str(taxonomy_map.get(word, "Not Found")) + ' '
        result += "\n"

    # save and output results
    output_filename = output_path + "topic_breakdown.txt"
    with open(output_filename, 'w') as f:
        f.write(result)

    print(f"Topic breakdown saved to {output_filename}")
    print(result)

In [ ]:
def topic_dist_plot(model, dictionary, taxonomy_file_name, input_taxonomy, output_taxonomy, output_path):
    taxonomy = pd.read_csv(taxonomy_file_name)
    topic_data = []

    taxonomy_map = dict(zip(taxonomy[input_taxonomy], taxonomy[output_taxonomy]))

    # LDA
    topic_ids = range(model.num_topics)
    get_terms = lambda i: model.get_topic_terms(i, topn=len(dictionary))

    # collect topic data
    for i in topic_ids:
        top_terms = get_terms(i)

        for term_id, importance in top_terms:
            word = str(dictionary[term_id]).strip() if term_id is not None else None

            # lookup taxonomy
            family_name = taxonomy_map.get(word, "Not Found") if word else "Not Found"

            topic_data.append({
                "Topic": f"Topic {i}",
                "Family": family_name,
                "Importance": importance
            })

    
    print("topic_data sample:", topic_data[:5])
    print("Number of entries:", len(topic_data))

    # convert to dataframe
    df_plot = pd.DataFrame(topic_data)

    # aggregate importance values for each (Topic, Family)
    df_agg = df_plot.groupby(["Topic", output_taxonomy], as_index=False)["Importance"].sum()

    # bubble plot
    plt.figure(figsize=(10, 6))
    bubble = sns.scatterplot(
        data=df_agg,
        x="Topic",
        y=output_taxonomy,
        size="Importance",
        hue="Importance",
        sizes=(20, 1000),
        palette="viridis",
        edgecolor="black",
        alpha=0.7
    )

    plt.xticks(rotation=45)
    plt.xlabel("Topic")
    plt.ylabel("Taxonomic Family")
    plt.title("Bubble Plot of Taxonomic Families per Topic")
    plt.legend(title="Importance")
    plt.tight_layout()

    # save and show
    output_filename = output_path + "topic_dist_plot.png"
    plt.savefig(output_filename, dpi=300)
    plt.show()
    print(f"Bubble plot saved to {output_filename}")

Original graph

In [ ]:
body_sites = pd.read_csv('../metag_topic_modeling/data_sets/HMP_V13_participant_data.csv')

body_site_mapping = {site: idx for idx, site in enumerate(body_sites['HMP_BODY_SITE'].unique())}

body_site_ints = body_sites['HMP_BODY_SITE'].map(body_site_mapping)

In [ ]:
def bubblePlot2(df, model, dictionary, filename, corpus):
    
    # detect topic count
    if hasattr(model, "num_topics"):
        num_topics = model.num_topics  # LDA
    else:
        num_topics = len(model.show_topics(num_topics=-1, formatted=False))  # HDP

    # find topic distributions for each document
    topic_distributions = []
    for doc_bow in corpus:
        doc_topics = model.get_document_topics(doc_bow, minimum_probability=0)
        # Convert to full vector (pad to num_topics)
        topic_vector = np.zeros(num_topics)
        for topic_id, prob in doc_topics:
            if topic_id < num_topics:  # safeguard
                topic_vector[topic_id] = prob
        topic_distributions.append(topic_vector)

    topic_distributions = np.array(topic_distributions)

    # assign strongest topic
    strongest_topic_indices = topic_distributions.argmax(axis=1)
    body_sites['Strongest_Topic'] = strongest_topic_indices

    # find topic counts by site
    topic_counts_by_site = body_sites.groupby(['HMP_BODY_SITE', 'Strongest_Topic']).size().reset_index(name="Count")
    print(topic_counts_by_site)

    table_filename = f"results2/{str(model)}_table_gensim.csv"
    topic_counts_by_site.to_csv(table_filename, index=False)
    print(f"Table saved to {table_filename}")

    # bubble plot
    plt.figure(figsize=(12, 7))
    bubble = sns.scatterplot(
        data=topic_counts_by_site,
        x="Strongest_Topic",
        y="HMP_BODY_SITE",
        size="Count",
        hue="Count",
        sizes=(20, 1000),
        palette="viridis",
        edgecolor="black",
        alpha=0.7
    )

    plt.xticks(rotation=45)
    plt.xlabel("Strongest Topic")
    plt.ylabel("Body Site")
    plt.title("Bubble Plot of Topic Assignments by Body Site")
    plt.legend(title="Sample Count", loc='upper right')
    plt.tight_layout()

    output_filename = f"results2/{str(model)}_bubble_plot_gensim.png"
    plt.savefig(output_filename, dpi=300)
    plt.show()
    print(f"Bubble plot saved to {output_filename}")

In [ ]:
def outputTableandGraph(df, model, dictionary, filename, corpus):

    # detect topic count
    if hasattr(model, "num_topics"):
        num_topics = model.num_topics  # LDA
    else:
        num_topics = len(model.show_topics(num_topics=-1, formatted=False))  # HDP

    # compute topic distributions for each document
    topic_distributions = []
    for doc_bow in corpus:
        doc_topics = model.get_document_topics(doc_bow, minimum_probability=0)
        # convert to full vector
        topic_vector = np.zeros(num_topics)
        for topic_id, prob in doc_topics:
            if topic_id < num_topics:  # safeguard
                topic_vector[topic_id] = prob
        topic_distributions.append(topic_vector)
    topic_distributions = np.array(topic_distributions)

    # assign strongest topic
    strongest_topic_indices = topic_distributions.argmax(axis=1)
    body_sites['Strongest_Topic'] = strongest_topic_indices

    # aggregate by body site
    topic_counts_by_site = body_sites.groupby(['HMP_BODY_SITE', 'Strongest_Topic']).size().unstack(fill_value=0)
    print(topic_counts_by_site)

    # save table
    table_filename = f"results2/{str(model)}_table_gensim.csv"
    topic_counts_by_site.to_csv(table_filename)
    print(f"Table saved to {table_filename}")

    # map topics to integers for plotting
    LDA_mapping = {topic: idx for idx, topic in enumerate(body_sites['Strongest_Topic'].unique())}
    LDA_ints = body_sites['Strongest_Topic'].map(LDA_mapping)


    '''
    # plot
    custom_colors = ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'pink', 'brown', 'olive', 'cyan']
    cmap = ListedColormap(custom_colors)

    fig = plt.figure(1, figsize=(8, 8))
    plt.clf()
    scatter = plt.scatter(result[:, 0], result[:, 1], c=LDA_ints, cmap=cmap, s=15)
    plot_filename = f"results/{filename}_comp_plot_gensim.svg"
    plt.savefig(plot_filename)
    print(f"Plot saved to {plot_filename}")
    plt.show()
    '''

Function to find optimal component number for each taxonomic level by perplexity, returning a graph and a table at that level.

Example on our dataset (HMP OTU counts)

Functionality I'd like to add:
- Run over a specific subset of data, i.e. mouth data
- Run LDA at any taxonomic level